In [ ]:
import cv2
from ultralytics import YOLO
import time
import cvzone
import numpy as np
import serial

# Load video
cap = cv2.VideoCapture(r"C:\Users\EVO TECH\Downloads\1535674-hd_1920_1080_24fps.mp4")

# Load YOLO model
model = YOLO(r"C:\Users\EVO TECH\Desktop\fire2\fire.pt")

# Initialize serial communication
ser = serial.Serial('COM9', 9600)

while True:
    ret, frame = cap.read()
    if not ret:
        break


    # Convert to HSV color space
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_fire = np.array([5, 100, 100])    # orange-red
    upper_fire = np.array([25, 255, 255])   # yellow-orange

    fire_mask = cv2.inRange(hsv, lower_fire, upper_fire)

    contours, _ = cv2.findContours(fire_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 500:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cvzone.putTextRect(frame, 'Color Fire', [x, y - 10], scale=1, thickness=2)

    # --- YOLO DETECTION ---

    results = model(frame)
    color_detected = False


    for res in results:
        boxes = res.boxes
        for box in boxes:
            conf = float(box.conf[0])
            if conf > 0.5:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 5)
                cvzone.putTextRect(frame, f'Fire {conf:.2f}', [x1 + 8, y1 + 30], scale=1.5, thickness=3)
                color_detected = True

    # Send signal via serial if fire is detected
    if color_detected:
        ser.write(b'1')
    else:
        ser.write(b'0')

    cv2.imshow('color', frame)

    if cv2.waitKey(1) & 0xFF == ord('z'):
        break

cap.release()
cv2.destroyAllWindows()
